In [ ]:
# =============================================================================
# Telugu (tel_Telu) IndicCorpV2 — Sentence tokenization + BPE/WordPiece pipeline
# Designed for Google Colab. Run cell-by-cell (each `# %%` is a Colab cell).
# =============================================================================

# %%
# ---- Cell 1: Install dependencies ----
!pip install -q datasets tokenizers huggingface_hub
!pip install -q indic-nlp-library

In [ ]:
# %%
# ---- Cell 2: Imports & config ----
import os
import random
import json
from datasets import load_dataset
from indicnlp.tokenize import sentence_tokenize

random.seed(42)

WORKDIR = "/content/telugu_pipeline"
os.makedirs(WORKDIR, exist_ok=True)

LANG_CODE = "te"          # indic-nlp lang code for Telugu
HF_SPLIT = "tel_Telu"     # confirmed split name -> data/te.txt in IndicCorpV2

DEV_SIZE = 1000
TEST_SIZE = 1000
MAX_TRAIN_SIZE = 1_000_000          # largest training experiment size
POOL_SIZE = MAX_TRAIN_SIZE + DEV_SIZE + TEST_SIZE   # 1,002,000

TRAIN_SIZES = [100_000, 300_000, 500_000, 1_000_000]
VOCAB_SIZES = [20_000, 30_000, 50_000]


In [ ]:
# %%
# ---- Cell 3: Stream all 31M rows once, sentence-split, reservoir-sample ----
# We CANNOT hold 31M paragraphs (which explode into far more sentences) in RAM.
# So we do a single streaming pass and keep a uniform random reservoir of
# POOL_SIZE sentences. This is an unbiased sample drawn from every one of the
# 31M rows, without ever materializing the full corpus.

ds = load_dataset(
    "ai4bharat/IndicCorpV2",
    "indiccorp_v2",
    split=HF_SPLIT,
    streaming=True,
)

reservoir = []          # list of sentence strings
n_seen = 0               # total sentences encountered so far
n_rows_processed = 0

for row in ds:
    n_rows_processed += 1
    para = row["text"] if "text" in row else list(row.values())[0]
    if not para or not para.strip():
        continue

    sents = sentence_tokenize.sentence_split(para, lang=LANG_CODE)

    for s in sents:
        s = s.strip()
        if len(s.split()) < 2:      # drop degenerate 0/1-word "sentences"
            continue
        n_seen += 1
        if len(reservoir) < POOL_SIZE:
            reservoir.append(s)
        else:
            j = random.randint(0, n_seen - 1)
            if j < POOL_SIZE:
                reservoir[j] = s

    if n_rows_processed % 500_000 == 0:
        print(f"rows processed: {n_rows_processed:,} | sentences seen: {n_seen:,} "
              f"| reservoir filled: {len(reservoir):,}")

print("DONE")
print(f"Total rows processed (should be ~31M): {n_rows_processed:,}")
print(f"Total sentences seen across corpus: {n_seen:,}")
print(f"Reservoir pool size: {len(reservoir):,}")


README.md:   0%|          | 0.00/4.01k [00:00<?, ?B/s]

DONE
Total rows processed (should be ~31M): 42,916,522
Total sentences seen across corpus: 73,253,467
Reservoir pool size: 1,002,000


In [ ]:
# %%
# ---- Cell 4: Build length-stratified dev/test sets from the pool ----
# "Different lengths, not similar" -> bucket by word count, sample evenly
# across buckets for both dev and test, so short/medium/long sentences are
# all represented instead of dev/test clustering around one length.

def length_bucket(s):
    n = len(s.split())
    if n <= 5:
        return "very_short"
    elif n <= 10:
        return "short"
    elif n <= 20:
        return "medium"
    elif n <= 35:
        return "long"
    else:
        return "very_long"

buckets = {}
for s in reservoir:
    buckets.setdefault(length_bucket(s), []).append(s)

for k, v in buckets.items():
    random.shuffle(v)

bucket_names = list(buckets.keys())
per_bucket_dev = DEV_SIZE // len(bucket_names)
per_bucket_test = TEST_SIZE // len(bucket_names)

dev_set, test_set = [], []
for name in bucket_names:
    pool = buckets[name]
    dev_take = pool[:per_bucket_dev]
    test_take = pool[per_bucket_dev:per_bucket_dev + per_bucket_test]
    dev_set.extend(dev_take)
    test_set.extend(test_take)
    buckets[name] = pool[per_bucket_dev + per_bucket_test:]  # remove used ones

# top up dev/test to exact sizes if rounding left them short
remaining_pool = [s for v in buckets.values() for s in v]
random.shuffle(remaining_pool)
while len(dev_set) < DEV_SIZE:
    dev_set.append(remaining_pool.pop())
while len(test_set) < TEST_SIZE:
    test_set.append(remaining_pool.pop())

train_pool = remaining_pool
random.shuffle(train_pool)

print(f"dev: {len(dev_set)}, test: {len(test_set)}, train_pool: {len(train_pool)}")
print("Dev length distribution:", {b: sum(1 for s in dev_set if length_bucket(s)==b) for b in bucket_names})
print("Test length distribution:", {b: sum(1 for s in test_set if length_bucket(s)==b) for b in bucket_names})

dev: 1000, test: 1000, train_pool: 1000000
Dev length distribution: {'short': 200, 'medium': 200, 'very_short': 200, 'long': 200, 'very_long': 200}
Test length distribution: {'short': 200, 'medium': 200, 'very_short': 200, 'long': 200, 'very_long': 200}


In [ ]:

# %%
# ---- Cell 5: Persist splits to disk ----
def write_lines(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        for l in lines:
            f.write(l.replace("\n", " ").strip() + "\n")

write_lines(f"{WORKDIR}/dev.txt", dev_set)
write_lines(f"{WORKDIR}/test.txt", test_set)
write_lines(f"{WORKDIR}/train_pool_full.txt", train_pool)  # up to MAX_TRAIN_SIZE lines

print("Saved dev.txt, test.txt, train_pool_full.txt to", WORKDIR)

Saved dev.txt, test.txt, train_pool_full.txt to /content/telugu_pipeline


In [ ]:
# %%
# ---- Cell 6: BPE / WordPiece training + tokenization functions ----
from tokenizers import Tokenizer, normalizers, pre_tokenizers
from tokenizers.models import BPE, WordPiece
from tokenizers.trainers import BpeTrainer, WordPieceTrainer

def build_tokenizer(kind, vocab_size):
    if kind == "bpe":
        tok = Tokenizer(BPE(unk_token="[UNK]"))
        trainer = BpeTrainer(
            vocab_size=vocab_size,
            special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
        )
    elif kind == "wordpiece":
        tok = Tokenizer(WordPiece(unk_token="[UNK]"))
        trainer = WordPieceTrainer(
            vocab_size=vocab_size,
            special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
        )
    else:
        raise ValueError(kind)

    tok.normalizer = normalizers.NFC()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    return tok, trainer

def train_and_tokenize(train_file, kind, vocab_size, out_dir):
    tok, trainer = build_tokenizer(kind, vocab_size)
    tok.train([train_file], trainer)
    os.makedirs(out_dir, exist_ok=True)
    save_path = f"{out_dir}/{kind}_vocab{vocab_size}.json"
    tok.save(save_path)

    def tokenize_file(path):
        with open(path, encoding="utf-8") as f:
            lines = [l.strip() for l in f if l.strip()]
        encoded = [tok.encode(l).tokens for l in lines]
        return encoded

    dev_tok = tokenize_file(f"{WORKDIR}/dev.txt")
    test_tok = tokenize_file(f"{WORKDIR}/test.txt")
    return tok, dev_tok, test_tok, save_path

In [ ]:
# %%
# ---- Cell 7: Run all (training_size x vocab_size) combos for BPE + WordPiece ----
results = []  # collect metadata for every run

for train_size in TRAIN_SIZES:
    subset = train_pool[:train_size]
    train_file = f"{WORKDIR}/train_{train_size}.txt"
    write_lines(train_file, subset)

    for vocab_size in VOCAB_SIZES:
        for kind in ["bpe", "wordpiece"]:
            out_dir = f"{WORKDIR}/tokenizers/{kind}/size_{train_size}_vocab_{vocab_size}"
            tok, dev_tok, test_tok, save_path = train_and_tokenize(
                train_file, kind, vocab_size, out_dir
            )

            avg_dev_tokens = sum(len(t) for t in dev_tok) / len(dev_tok)
            avg_test_tokens = sum(len(t) for t in test_tok) / len(test_tok)

            results.append({
                "kind": kind,
                "train_size": train_size,
                "vocab_size": vocab_size,
                "tokenizer_path": save_path,
                "avg_tokens_per_dev_sentence": avg_dev_tokens,
                "avg_tokens_per_test_sentence": avg_test_tokens,
            })

            print(f"[{kind}] train_size={train_size:>8,} vocab={vocab_size:>6,} "
                  f"| avg dev tok/sent={avg_dev_tokens:.2f} "
                  f"| avg test tok/sent={avg_test_tokens:.2f}")

[bpe] train_size= 100,000 vocab=20,000 | avg dev tok/sent=32.11 | avg test tok/sent=33.44
[wordpiece] train_size= 100,000 vocab=20,000 | avg dev tok/sent=33.14 | avg test tok/sent=34.70
[bpe] train_size= 100,000 vocab=30,000 | avg dev tok/sent=30.27 | avg test tok/sent=31.54
[wordpiece] train_size= 100,000 vocab=30,000 | avg dev tok/sent=31.15 | avg test tok/sent=32.57
[bpe] train_size= 100,000 vocab=50,000 | avg dev tok/sent=28.52 | avg test tok/sent=29.64
[wordpiece] train_size= 100,000 vocab=50,000 | avg dev tok/sent=29.14 | avg test tok/sent=30.33
[bpe] train_size= 300,000 vocab=20,000 | avg dev tok/sent=32.06 | avg test tok/sent=33.40
[wordpiece] train_size= 300,000 vocab=20,000 | avg dev tok/sent=33.19 | avg test tok/sent=34.69
[bpe] train_size= 300,000 vocab=30,000 | avg dev tok/sent=30.20 | avg test tok/sent=31.39
[wordpiece] train_size= 300,000 vocab=30,000 | avg dev tok/sent=31.11 | avg test tok/sent=32.42
[bpe] train_size= 300,000 vocab=50,000 | avg dev tok/sent=28.33 | avg 

In [ ]:
# %%
# ---- Cell 8: Save summary + example tokenized sentences ----
with open(f"{WORKDIR}/results_summary.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# Show a worked example: same dev sentence tokenized by every BPE tokenizer
sample_sentence = dev_set[0]
print("Sample dev sentence:", sample_sentence)
for train_size in TRAIN_SIZES:
    for vocab_size in VOCAB_SIZES:
        path = f"{WORKDIR}/tokenizers/bpe/size_{train_size}_vocab_{vocab_size}/bpe_vocab{vocab_size}.json"
        tok = Tokenizer.from_file(path)
        toks = tok.encode(sample_sentence).tokens
        print(f"  train={train_size:>8,} vocab={vocab_size:>6,} -> {toks}")

print("\nAll tokenizers, splits, and results_summary.json saved under:", WORKDIR)

Sample dev sentence: 'టీ' లో పురుగుల మందు కలిపిన బాలిక. .
  train= 100,000 vocab=20,000 -> ["'", 'టీ', "'", 'లో', 'పురుగుల', 'మందు', 'కలిపిన', 'బాలిక', '.', '.']
  train= 100,000 vocab=30,000 -> ["'", 'టీ', "'", 'లో', 'పురుగుల', 'మందు', 'కలిపిన', 'బాలిక', '.', '.']
  train= 100,000 vocab=50,000 -> ["'", 'టీ', "'", 'లో', 'పురుగుల', 'మందు', 'కలిపిన', 'బాలిక', '.', '.']
  train= 300,000 vocab=20,000 -> ["'", 'టీ', "'", 'లో', 'పురుగుల', 'మందు', 'కలిపిన', 'బాలిక', '.', '.']
  train= 300,000 vocab=30,000 -> ["'", 'టీ', "'", 'లో', 'పురుగుల', 'మందు', 'కలిపిన', 'బాలిక', '.', '.']
  train= 300,000 vocab=50,000 -> ["'", 'టీ', "'", 'లో', 'పురుగుల', 'మందు', 'కలిపిన', 'బాలిక', '.', '.']
  train= 500,000 vocab=20,000 -> ["'", 'టీ', "'", 'లో', 'పురుగుల', 'మందు', 'కలి', 'పిన', 'బాలిక', '.', '.']
  train= 500,000 vocab=30,000 -> ["'", 'టీ', "'", 'లో', 'పురుగుల', 'మందు', 'కలిపిన', 'బాలిక', '.', '.']
  train= 500,000 vocab=50,000 -> ["'", 'టీ', "'", 'లో', 'పురుగుల', 'మందు', 'కలిపిన', 'బాలిక', '.', '.']
  